In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

In [2]:
def compute_modes_for_energy_thresholds(projectnames, thresholds):
    """
    For each project, compute the number of modes required to reach
    specified cumulative energy thresholds.

    Parameters
    ----------
    projectnames : list[str]
    thresholds : list[float]
        e.g. [0.95, 0.9, 0.8]
    """

    print("\n=== Modes Required for Energy Thresholds ===\n")

    for proj in projectnames:

        S = np.loadtxt(f"{proj}_SV.txt")

        energy = S**2
        cumulative_energy = np.cumsum(energy) / np.sum(energy)

        print(f"Project: {proj}")

        for thresh in thresholds:
            idx = np.searchsorted(cumulative_energy, thresh) + 1  # +1 for 1-based mode count
            print(f"  Modes for {thresh*100:.1f}% energy: {idx}")

        print("")

In [ ]:
def compute_shannon_entropy_POD(projectnames, outfile="ShannonEntropy.txt"):
    """
    Compute normalized Shannon entropy of singular value spectra
    for each project, and save all entropy values to a single text file.

    Parameters
    ----------
    projectnames : list of str
        List of project names.
    outfile : str
        Name of output text file.
    """

    print("\n=== Shannon Entropy of Singular Value Spectra ===\n")

    entropy_vals = []

    for proj in projectnames:

        S = np.loadtxt(f"{proj}_SV.txt")

        lam = S**2
        total = np.sum(lam)

        p = lam / total

        # Avoid log(0)
        p = p[p > 0]

        n = len(S)

        H = -np.sum(p * np.log(p)) / np.log(n)

        entropy_vals.append(H)

        print(f"Project: {proj}")
        print(f"  Shannon Entropy: {H:.6f}\n")

    np.savetxt(outfile, np.array(entropy_vals), fmt="%.10f")
    print(f"Saved Shannon entropy values to {outfile}")

In [ ]:
def compute_dmd_energy(projectNames, save_normalized=True, save_cumulative=True):
    """
    Computes DMD modal energies using:
        E_j = sum_k |g_j(t_k)|^2
    where g_j(t_k) comes from BT = diag(b) @ T

    Energies are sorted in descending order before computing
    normalized and cumulative values.

    Parameters
    ----------
    projectNames : list of str
        List of project name prefixes
    save_normalized : bool
        Whether to also save normalized energies
    save_cumulative : bool
        Whether to also save cumulative energies
    """

    for project in projectNames:

        print(f"\nProcessing project: {project}")

        # -----------------------
        # Load data
        # -----------------------
        b_path = f"{project}_bval.txt"
        eig_path = f"{project}_DMDeig.txt"

        b = np.loadtxt(b_path, dtype=complex)
        lam = np.loadtxt(eig_path, dtype=complex)

        # Ensure 1D arrays
        b = np.atleast_1d(b)
        lam = np.atleast_1d(lam)

        r = len(b)

        # -----------------------
        # Determine number of snapshots
        # -----------------------
        # Assumption: number of time steps = number of modes
        m = r

        # -----------------------
        # Construct Vandermonde matrix
        # -----------------------
        # T[j, k] = lambda_j^k
        powers = np.arange(m)
        T = lam[:, None] ** powers[None, :]   # shape (r, m)

        # -----------------------
        # Compute BT
        # -----------------------
        BT = (b[:, None]) * T   # diag(b) @ T

        # -----------------------
        # Compute energies
        # -----------------------
        energies = np.sum(np.abs(BT)**2, axis=1)

        # -----------------------
        # Save raw energies (unsorted)
        # -----------------------
        out_path = f"{project}_dmdEnergies.txt"
        np.savetxt(out_path, energies.real)
        print(f"Saved energies to {out_path}")

        # -----------------------
        # Sort energies (descending)
        # -----------------------
        sort_idx = np.argsort(energies)[::-1]
        energies_sorted = energies[sort_idx]

        # -----------------------
        # Normalize energies
        # -----------------------
        total_energy = np.sum(energies_sorted)
        norm_energies = energies_sorted / total_energy

        if save_normalized:
            out_path_norm = f"{project}_dmdEnergies_normalized.txt"
            np.savetxt(out_path_norm, norm_energies.real)
            print(f"Saved normalized energies to {out_path_norm}")

        # -----------------------
        # Compute cumulative energy
        # -----------------------
        if save_cumulative:
            cumulative = np.cumsum(norm_energies)

            out_path_cum = f"{project}_dmdEnergies_cumulative.txt"
            np.savetxt(out_path_cum, cumulative.real)
            print(f"Saved cumulative energies to {out_path_cum}")

        # -----------------------
        # (Optional) Save sorting indices
        # -----------------------
        sort_idx_path = f"{project}_dmdEnergy_sortIdx.txt"
        np.savetxt(sort_idx_path, sort_idx, fmt='%d')
        print(f"Saved sorting indices to {sort_idx_path}")

In [ ]:
def compute_shannon_entropy_DMD(projectnames, outfile="ShannonEntropy_DMD.txt"):
    """
    Compute normalized Shannon entropy of DMD energy spectra
    for each project, using precomputed normalized energies.

    Parameters
    ----------
    projectnames : list of str
        List of project names.
    outfile : str
        Name of output text file.
    """

    print("\n=== Shannon Entropy of DMD Energy Spectra ===\n")

    entropy_vals = []

    for proj in projectnames:

        # -----------------------
        # Load normalized DMD energies
        # -----------------------
        p = np.loadtxt(f"{proj}_dmdEnergies_normalized.txt")

        # Ensure array
        p = np.atleast_1d(p)

        # Remove zeros to avoid log issues
        p = p[p > 0]

        n = len(p)

        # -----------------------
        # Compute Shannon entropy
        # -----------------------
        H = -np.sum(p * np.log(p)) / np.log(n)

        entropy_vals.append(H)

        print(f"Project: {proj}")
        print(f"  Shannon Entropy (DMD): {H:.6f}\n")

    # -----------------------
    # Save results
    # -----------------------
    np.savetxt(outfile, np.array(entropy_vals), fmt="%.10f")
    print(f"Saved Shannon entropy values to {outfile}")

In [5]:
if __name__ == "__main__":

    projects = ["Valve_0.3mm", "Valve_0.5mm", "Valve_0.75mm"]

    thresholds = [0.999, 0.995, 0.99, 0.95]

    compute_modes_for_energy_thresholds(projects, thresholds)

    compute_shannon_entropy_POD(projects, outfile="ShannonEntropy_POD.txt")


=== Modes Required for Energy Thresholds ===

Project: Valve_0.3mm
  Modes for 99.9% energy: 44
  Modes for 99.5% energy: 32
  Modes for 99.0% energy: 26
  Modes for 95.0% energy: 5

Project: Valve_0.5mm
  Modes for 99.9% energy: 45
  Modes for 99.5% energy: 35
  Modes for 99.0% energy: 29
  Modes for 95.0% energy: 9

Project: Valve_0.75mm
  Modes for 99.9% energy: 51
  Modes for 99.5% energy: 42
  Modes for 99.0% energy: 37
  Modes for 95.0% energy: 22


=== Shannon Entropy of Singular Value Spectra ===

Project: Valve_0.3mm
  Shannon Entropy: 0.111306

Project: Valve_0.5mm
  Shannon Entropy: 0.137225

Project: Valve_0.75mm
  Shannon Entropy: 0.323073

Saved Shannon entropy values to ShannonEntropy_POD.txt
